# K5 s2024 on DAD, then the K5 vs LaGoVAD table

Reads the checkpoint K5 s2024 already on Drive on the DAD bake `zs_dad.ipynb` built. **Nothing trains and nothing is extracted.** DoTA-CAP-dev, DADA-2000 and TAD are already read. Upload branch `v_final_las_vad` to `Thesis-Final/kat-vad` first.

| step | what | GPU | estimate |
|---|---|---|---|
| 1-3 | setup, preflight (checkpoint, J10, K5's config, bake, labels) | no | 2 min |
| 4 | `core.evaluate`: K5 on DAD (whole video, s7, raw scores) -> `outputs/kim/eval/dad/K5/s2024` | yes (light) | 5 min |
| 5 | `core.tools.k5_zs_report` -> **paste `REPORTS/k5_zs_report.md` to Claude** | no | 1-2 min |

A re-run costs seconds: step 4 skips on its `results.json`. One seed: **PRELIMINARY (n = 1 seed)**. Rule for the DAD read: `core/docs/final/ZS_DAD.md`.

In [ ]:
from google.colab import drive

drive.mount("/content/drive")

In [ ]:
import json
import os
import shutil
import subprocess
import sys
from pathlib import Path

DRIVE_F = '/content/drive/MyDrive/Thesis-Final'   # code, DAD bake and labels, K5's run, every output
os.environ['PROJECT_ROOT']       = DRIVE_F
os.environ['KATVAD_DATA_ROOT']   = '/content/drive/MyDrive/Thesis/data'
os.environ['KATVAD_CACHE_ROOT']  = f'{DRIVE_F}/cache'
os.environ['KATVAD_CKPT_ROOT']   = '/content/drive/MyDrive/Thesis/ckpts'
os.environ['KATVAD_OUTPUT_ROOT'] = f'{DRIVE_F}/outputs'

os.environ['REPO'] = f'{DRIVE_F}/kat-vad'
REPO = Path(os.environ['REPO'])
assert (REPO / 'core' / 'constants.py').is_file(), f'no code at {REPO} -- upload branch v_final_las_vad there'
assert (REPO / 'core/tools/k5_zs_report.py').is_file(), 'this upload has no k5_zs_report -- re-upload kat-vad/'
os.environ['PYTHONPATH'] = str(REPO)
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))
for name in [m for m in sys.modules if m == 'core' or m.startswith('core.')]:
    del sys.modules[name]              # constants reads KATVAD_* at import time
from core import constants  # noqa: E402

assert constants.REPO_ROOT == REPO, f'imported core from {constants.REPO_ROOT}, expected {REPO}'

ARM, SEED = constants.KIM_MAIN_ARM, constants.KIM_SEED     # K5, 2024
DSET, STRIDE = constants.DAD_DATASET, constants.DAD_STRIDE  # DAD, 7
ENC      = constants.V5_ENCODER                             # V2-S
KIM      = constants.OUTPUT_ROOT / constants.KIM_OUTPUT_DIRNAME
RUN      = KIM / ARM / f's{SEED}'
CKPT     = RUN / 'checkpoint_last.pt'
LAGOVAD  = constants.OUTPUT_ROOT / constants.ZS_LAGOVAD_OUTPUT_DIRNAME
OUT      = KIM / 'eval' / constants.DAD_OUTPUT_SET / ARM / f's{SEED}'
REPORTS  = KIM / 'REPORTS'
BAKE     = constants.CACHE_ROOT / 'v2' / 'A2_S' / f'{DSET}_s{STRIDE}_nexar'
DATA     = Path(DRIVE_F) / 'data' / f'{DSET}_s{STRIDE}'
LOCAL    = Path('/content/k5_dad')
LOCAL.mkdir(parents=True, exist_ok=True)

for name, path in (('K5 checkpoint', CKPT), ('K5 verified.json', RUN / 'verified.json'), ('K5 metrics', RUN / 'metrics.jsonl'),
                   ('K5 config', RUN / 'config.yaml'), ('DAD bake', BAKE / constants.V2_INPUT_MANIFEST_FILENAME),
                   ('DAD labels', DATA / constants.FRAME_LABELS_TEST_FILENAME),
                   ('LaGoVAD DAD read', LAGOVAD / 'eval' / constants.DAD_OUTPUT_SET / constants.ZS_LAGOVAD_ARM / constants.ZS_LAGOVAD_RUN / 'results.json'),
                   ('K5 TAD read', KIM / 'eval' / 'tad' / ARM / f's{SEED}' / 'results.json'),
                   ('K5 DADA read', KIM / 'eval' / 'dada' / ARM), ('K5 DoTA read', KIM / 'eval' / 'dota' / ARM)):
    assert path.exists(), f'missing {name}: {path}'
    print(f'  {name:18s} {path}')

In [ ]:
%%bash
pip install -q "transformers==4.56.*" av einops faiss-cpu scikit-learn openpyxl
python -c "import av, einops, faiss, sklearn; print('deps OK')"
nvidia-smi --query-gpu=name,memory.total --format=csv,noheader || echo 'NO GPU -- step 4 will be slow'

In [ ]:
import yaml

ENV = dict(os.environ, USE_TF='0', TF_CPP_MIN_LOG_LEVEL='2')   # transformers must not import TF


def run(cmd, log=None):
    """Stream a child's output here (and to ``log``); raise with its tail on failure."""
    tail = []
    sink = open(log, 'a') if log else None
    try:
        with subprocess.Popen([str(c) for c in cmd], cwd=str(REPO), env=ENV, stdout=subprocess.PIPE,
                              stderr=subprocess.STDOUT, text=True, bufsize=1) as proc:
            for line in proc.stdout:
                tail = (tail + [line])[-40:]
                if sink:
                    sink.write(line)
                print(line, end='')
    finally:
        if sink:
            sink.close()
    if proc.returncode:
        print('\n' + '=' * 30 + f'\nFAILED (exit {proc.returncode}) -- paste these lines to Claude:\n' + ''.join(tail))
        raise RuntimeError(f'exit {proc.returncode}: {" ".join(str(c) for c in cmd[:4])} ...')


def publish(src, dst):
    """VM-local result -> Drive ATOMICALLY: ``<dst>.partial``, fsync, rename (C10)."""
    if src.is_dir():
        for f in [p for p in src.rglob('*') if p.is_file()]:
            publish(f, dst / f.relative_to(src))
        return
    dst.parent.mkdir(parents=True, exist_ok=True)
    tmp = dst.with_name(dst.name + '.partial')
    shutil.copy2(src, tmp)
    with open(tmp, 'rb+') as fh:
        os.fsync(fh.fileno())
    os.replace(tmp, dst)


def stage(src, dst):
    """A whole Drive dir to VM disk once (bit-identical; C2)."""
    if not dst.exists():
        tmp = dst.with_name(dst.name + '.partial')
        shutil.rmtree(tmp, ignore_errors=True)
        shutil.copytree(src, tmp)
        os.replace(tmp, dst)
    return dst


# Preflight on small files only (C39): J10 from the record written when the checkpoint was verified.
verified = json.loads((RUN / 'verified.json').read_text())
lines = [l for l in (RUN / 'metrics.jsonl').read_text().splitlines() if l.strip()]
last_step = json.loads(lines[-1])['global_step']
assert verified['step'] == last_step, f'J10: verified step {verified["step"]} != metrics {last_step}'
assert CKPT.stat().st_size == verified['ckpt_bytes'], f'checkpoint size {CKPT.stat().st_size} != verified {verified["ckpt_bytes"]}'
cfg = yaml.safe_load((RUN / 'config.yaml').read_text())
want = {'enabled': True, 'prototypes': True, 'edge_mask': True}
assert {k: cfg['kim'][k] for k in want} == want, f'not K5: {cfg["kim"]}'
assert cfg['dvs']['theta'] == 1.0 and cfg['train']['seed'] == SEED and cfg['kip']['enabled'] is False, 'not K5 s2024'
manifest = json.loads((BAKE / constants.V2_INPUT_MANIFEST_FILENAME).read_text())
assert (manifest['crn'], manifest['motion']) == (constants.V2_OFF, ENC), manifest
labels = json.loads((DATA / constants.FRAME_LABELS_TEST_FILENAME).read_text())
assert manifest['baked_ids'] == len(labels) == constants.DAD_TEST_VIDEOS, (manifest['baked_ids'], len(labels))
print(f'{ARM} s{SEED}: step {last_step} (J10 OK), kim {want}, DVS off | bake A2 (no CRN, {ENC}), {len(labels)} videos')

## 4. Score K5 on DAD (whole video, s7, raw scores, `--save-scores`)

In [ ]:
if (OUT / 'results.json').exists():
    print(f'{ARM} on DAD: already read')
else:
    local = LOCAL / 'eval'
    shutil.rmtree(local, ignore_errors=True)
    run([sys.executable, '-m', 'core.evaluate', '--ckpt', CKPT, '--set', f'data.dataset={DSET}',
         '--data-dir', stage(DATA, LOCAL / 'data'), '--clip-dir', stage(BAKE, LOCAL / 'bake'),
         '--score-norm', 'auto', '--save-scores', '--output-dir', local], log=LOCAL / 'eval.log')
    publish(local, OUT)
    drive.flush_and_unmount()
    drive.mount('/content/drive')
r = json.loads((OUT / 'results.json').read_text())
assert r['score_norm'] == 'none' and r['num_videos'] == constants.DAD_TEST_VIDEOS, r
print(f"{ARM} on DAD: macro {r['auc_macro']:.4f} over {r['auc_macro_videos']} | micro {r['auc']:.4f} | AP {r['ap']:.4f}")

## 5. The table -> **paste `REPORTS/k5_zs_report.md` to Claude**

In [ ]:
local = LOCAL / 'report'
run([sys.executable, '-m', 'core.tools.k5_zs_report', '--lagovad-root', LAGOVAD, '--kim-root', KIM, '--seed', SEED,
     '--out-dir', local])
publish(local, REPORTS)
print((REPORTS / 'k5_zs_report.md').read_text())
drive.flush_and_unmount()          # C10: every pending Drive upload lands before the VM goes
print('Drive flushed -- paste REPORTS/k5_zs_report.md to Claude')